In [ ]:
import snowflake.connector
import pandas as pd
import os

# Connect to Snowflake
conn = snowflake.connector.connect(
    user=os.environ.get('SNOWFLAKE_USER'),
    password=os.environ.get('SNOWFLAKE_PASSWORD'),
    account='apc44937.us-east-1',
    warehouse='FINTECH_WH',
    database='FINTECH_DB',
    schema='PROD'
)

# SQL query
query = """
SELECT *
FROM FCT_CUSTOMER_FEATURES
"""

# Load data into pandas dataframe
df = pd.read_sql(query, conn)
customer_ids = df['CUSTOMER_ID'].reset_index(drop=True)

# Verify data loaded correctly
print("Data Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

# Close connection
conn.close()

C:\Users\deepe\AppData\Local\Temp\ipykernel_16788\3613729667.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data Shape: (6353307, 14)

First 5 Rows:
   CUSTOMER_ID  TOTAL_TRANSACTIONS  TOTAL_AMOUNT_SENT  AVG_TRANSACTION_AMOUNT  \
0   C206807223                   1           35754.58                35754.58   
1   C494450505                   1           22342.86                22342.86   
2  C1683497934                   1          136691.46               136691.46   
3  C1523467022                   1           18664.46                18664.46   
4   C145351124                   1           16769.75                16769.75   

   MAX_TRANSACTION_AMOUNT  UNIQUE_RECIPIENTS  FIRST_TRANSACTION_HOUR  \
0                35754.58                  1                      22   
1                22342.86                  1                      35   
2               136691.46                  1                      36   
3                18664.46                  1                      36   
4                16769.75                  1                     254   

   LAST_TRANSACTION_HOUR  FRAUD_TRANSAC

In [2]:
from sklearn.preprocessing import StandardScaler

df = df.drop(columns = ['CUSTOMER_ID'])

# fill null values in transaction_frequency_rate
df['TRANSACTION_FREQUENCY_RATE'] = (
    df['TRANSACTION_FREQUENCY_RATE'].fillna(0)
)

# create target variable (the thing that needs to be predicted)
df['is_churned'] = (df['RECENCY_SCORE'] > 700).astype(int)

# Select ML features
feature_cols = [
    'TOTAL_TRANSACTIONS',
    'TOTAL_AMOUNT_SENT',
    'AVG_TRANSACTION_AMOUNT',
    'ACTIVE_HOURS_SPAN',
    'TRANSACTION_FREQUENCY_RATE',
    'IS_HIGH_VALUE'
]


X = df[feature_cols]
y = df['is_churned']

# 5. Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for readability
X_scaled = pd.DataFrame(
    X_scaled,
    columns=feature_cols
)

# 6. Print churn distribution
print("Churn Distribution:")
print(y.value_counts())

print("\nChurn Percentage:")
print(y.value_counts(normalize=True) * 100)

# 7. Verify output
print("\nScaled Feature Shape:", X_scaled.shape)
print("\nFirst 5 Rows of Scaled Features:")
print(X_scaled.head())

Churn Distribution:
is_churned
0    5410283
1     943024
Name: count, dtype: int64

Churn Percentage:
is_churned
0    85.156958
1    14.843042
Name: proportion, dtype: float64

Scaled Feature Shape: (6353307, 6)

First 5 Rows of Scaled Features:
   TOTAL_TRANSACTIONS  TOTAL_AMOUNT_SENT  AVG_TRANSACTION_AMOUNT  \
0           -0.038253          -0.238891               -0.238817   
1           -0.038253          -0.261083               -0.261044   
2           -0.038253          -0.071871               -0.071537   
3           -0.038253          -0.267170               -0.267140   
4           -0.038253          -0.270305               -0.270280   

   ACTIVE_HOURS_SPAN  TRANSACTION_FREQUENCY_RATE  IS_HIGH_VALUE  
0          -0.030172                   -0.011099      -0.646716  
1          -0.030172                   -0.011099      -0.646716  
2          -0.030172                   -0.011099      -0.646716  
3          -0.030172                   -0.011099      -0.646716  
4          -0.0

In [3]:
from sklearn.cluster import KMeans

# Create KMeans model
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

# Fit model and assign clusters
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataframe
df['cluster'] = clusters

# Check results
print(df[['cluster']].head())
print(df['cluster'].value_counts())

# Mean values per cluster to understand what each cluster represents
cluster_profile = df.groupby('cluster')[feature_cols].mean().round(2)
print("\nCluster Profiles:")
print(cluster_profile)

   cluster
0        0
1        0
2        0
3        0
4        0
cluster
0    4473120
1    1870889
3       5099
2       4199
Name: count, dtype: int64

Cluster Profiles:
         TOTAL_TRANSACTIONS  TOTAL_AMOUNT_SENT  AVG_TRANSACTION_AMOUNT  \
cluster                                                                  
0                       1.0           50940.33                50940.33   
1                       1.0          488068.43               488068.43   
2                       2.0          352675.47               176093.74   
3                       2.0          378113.24               189017.93   

         ACTIVE_HOURS_SPAN  TRANSACTION_FREQUENCY_RATE  IS_HIGH_VALUE  
cluster                                                                
0                     0.00                        0.00           0.00  
1                     0.00                        0.00           1.00  
2                   266.99                        0.01           0.30  
3                    69.

In [4]:
# Create mapping dictionary
cluster_mapping = {
    0: 'Low Value One-Time',
    1: 'High Value One-Time',
    2: 'Repeat High Value',
    3: 'Repeat Mid Value'
}

# Add segment names
df['segment'] = df['cluster'].map(cluster_mapping)

# Check segment distribution
print("Segment Distribution:")
print(df['segment'].value_counts())

# Verify cluster-to-segment mapping
print("\nCluster-Segment Mapping:")
print(df[['cluster', 'segment']].head(10))

Segment Distribution:
segment
Low Value One-Time     4473120
High Value One-Time    1870889
Repeat Mid Value          5099
Repeat High Value         4199
Name: count, dtype: int64

Cluster-Segment Mapping:
   cluster              segment
0        0   Low Value One-Time
1        0   Low Value One-Time
2        0   Low Value One-Time
3        0   Low Value One-Time
4        0   Low Value One-Time
5        0   Low Value One-Time
6        1  High Value One-Time
7        0   Low Value One-Time
8        0   Low Value One-Time
9        0   Low Value One-Time


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

# 2. Create Logistic Regression model
model = LogisticRegression(random_state=42, max_iter = 1000,  class_weight='balanced')

# 3. Train model
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)

# 5. Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# 6. Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print(f"Model Accuracy: {accuracy*100:.2f}%")

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.74      0.79   1082127
           1       0.16      0.28      0.20    188535

    accuracy                           0.67   1270662
   macro avg       0.51      0.51      0.50   1270662
weighted avg       0.75      0.67      0.71   1270662


Model Accuracy: 0.6711
Model Accuracy: 67.11%


In [6]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from sklearn.linear_model import LogisticRegression

# 1. Get churn probabilities for all customers
churn_proba = model.predict_proba(X_scaled)[:, 1]

# 2. Build results dataframe
results_df = pd.DataFrame({
    'CUSTOMER_ID': customer_ids.values,
    'SEGMENT': df['segment'].values,
    'IS_CHURNED': y.values,
    'CHURN_PROBABILITY': churn_proba
})

print("Results Shape:", results_df.shape)
print(results_df.head())

# 3. Write back to Snowflake
conn2 = snowflake.connector.connect(
    user='Deepesh',
    password='DeepeshGorai@123',
    account='apc44937.us-east-1',
    warehouse='FINTECH_WH',
    database='FINTECH_DB',
    schema='PROD'
)

success, nchunks, nrows, _ = write_pandas(
    conn2,
    results_df,
    'ML_CUSTOMER_SCORES',
    auto_create_table=True
)

print(f"\nWrite successful: {success}")
print(f"Rows written: {nrows}")

conn2.close()

Results Shape: (6353307, 4)
   CUSTOMER_ID             SEGMENT  IS_CHURNED  CHURN_PROBABILITY
0   C206807223  Low Value One-Time           1           0.494417
1   C494450505  Low Value One-Time           1           0.495154
2  C1683497934  Low Value One-Time           1           0.488873
3  C1523467022  Low Value One-Time           1           0.495356
4   C145351124  Low Value One-Time           0           0.495460

Write successful: True
Rows written: 6353307
